# Caso H · 02 Tools sobre InfluxDB

> _Tutorial · Caso de uso: **H — RAG + Chatbot** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Implementar `query_influxdb`, `compare_periods` y `get_building_state` con fallback mock. Probar con un par de preguntas del golden set.


## 2. Qué se aprende

- Estructura de una tool: input strict, output JSON.
- Cómo serializar respuestas de Flux.
- Cómo distinguir 'sin datos' de 'error'.


## 3. Contexto del caso de uso

Tools son el corazón del chatbot. Estables y rápidas.


## 4. Relación con CENTINELA+

Las tools también las consume el Dashboard Adapter.


## 5. Relación con Medallion

Lee plata.


## 6. Datos de entrada

InfluxDB (real o mock).


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

5 tags y `value`.


## 9. Carga de datos o mock

Cargamos un mock simple para offline.


In [2]:
df, _ = mocks.make_ingauge_aula01_mock(days=2)
df = df.set_index("timestamp")
df.head()


,Indoor_CO2,Indoor_Temp,Indoor_Hum,Indoor_Noise,Indoor_Lux,Occupied,People_Count,CoolingState
timestamp,,,,,,,,
2024-09-09 00:00:00+02:00,451.3,17.13,42.28,32.8,0.0,0,0,0
2024-09-09 00:01:00+02:00,407.7,16.46,40.75,32.7,21.9,0,0,0
2024-09-09 00:02:00+02:00,406.0,17.01,48.91,35.9,0.0,0,0,0
2024-09-09 00:03:00+02:00,401.5,17.49,45.57,36.3,20.7,0,0,0
2024-09-09 00:04:00+02:00,377.3,17.50,39.57,34.1,89.1,0,0,0


## 10. Exploración paso a paso

Implementamos las 3 tools.


In [3]:
def query_influxdb(variable: str, start: str = "-1d", aggregation: str = "mean",
                   asset_id: str = "AULA01") -> dict:
    """Query simple. Si no hay cliente, usa el mock In-Gauge."""
    client = get_influx_client()
    var_to_csv = {"co2": "Indoor_CO2", "temperature_01": "Indoor_Temp",
                  "luminosity": "Indoor_Lux", "people_count": "People_Count"}
    if client is None:
        col = var_to_csv.get(variable)
        if col is None:
            return {"error": f"variable {variable} no soportada en mock"}
        s = df[col]
        agg = {"mean": s.mean, "max": s.max, "min": s.min, "last": lambda: s.iloc[-1]}.get(aggregation, s.mean)
        return {"variable": variable, "asset_id": asset_id, "agg": aggregation,
                "value": float(agg()), "n": int(len(s)), "source": "mock"}
    flux = f'from(bucket:"telemetry") |> range(start:{start}) |> filter(fn:(r)=>r.variable=="{variable}" and r.asset_id=="{asset_id}") |> {aggregation}()'
    res = client.query_api().query_data_frame(flux, org=os.environ.get("INFLUXDB_ORG", "captia"))
    return {"variable": variable, "asset_id": asset_id, "agg": aggregation, "value": float(res["_value"].iloc[0]) if len(res) else None}

import os

print(query_influxdb("co2", aggregation="mean"))
print(query_influxdb("temperature_01", aggregation="max"))


{'variable': 'co2', 'asset_id': 'AULA01', 'agg': 'mean', 'value': 569.5137152777778, 'n': 2880, 'source': 'mock'}
{'variable': 'temperature_01', 'asset_id': 'AULA01', 'agg': 'max', 'value': 24.93, 'n': 2880, 'source': 'mock'}


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Tools 2 y 3.


In [4]:
def compare_periods(variable: str, p1: tuple[str, str], p2: tuple[str, str], aggregation: str = "mean") -> dict:
    return {
        "variable": variable,
        "p1": query_influxdb(variable, start=p1[0], aggregation=aggregation)["value"],
        "p2": query_influxdb(variable, start=p2[0], aggregation=aggregation)["value"],
        "diff": None,
    }

def get_building_state(asset_id: str = "AULA01") -> dict:
    res = {
        "asset_id": asset_id,
        "co2_last": query_influxdb("co2", aggregation="last", asset_id=asset_id)["value"],
        "temp_last": query_influxdb("temperature_01", aggregation="last", asset_id=asset_id)["value"],
    }
    return res

print(compare_periods("co2", ("-7d", "-1d"), ("-1d", "now")))
print(get_building_state())


{'variable': 'co2', 'p1': 569.5137152777778, 'p2': 569.5137152777778, 'diff': None}
{'asset_id': 'AULA01', 'co2_last': 376.0, 'temp_last': 17.36}


## 13. Visualizaciones explicativas

Demo plot del estado.


In [5]:
state = get_building_state()
pd.Series(state).drop("asset_id").plot.bar(color="#3F51B5", figsize=(6, 3))
plt.title("Estado AULA01 (mock)")
plt.tight_layout()


## 14. Validaciones

Output siempre dict serializable.


In [6]:
import json
json.dumps(query_influxdb("co2"))


'{"variable": "co2", "asset_id": "AULA01", "agg": "mean", "value": 569.5137152777778, "n": 2880, "source": "mock"}'

## 15. Errores comunes

1. Devolver pandas en vez de tipos primitivos — no serializa.
2. No envolver Flux exceptions.
3. No registrar la firma exacta para que el LLM la entienda.


## 16. Ejercicios propuestos

1. Añade `aggregation='median'`.
2. Implementa caché en Redis con `functools.lru_cache`.
3. Convierte a Pydantic models el output.


## 17. Cómo se reutiliza con datos reales

Cliente real conectado; el resto se mantiene.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `08_case_H_rag_chatbot/03_mock_tools_modelos_predictivos.ipynb`.
- Documento web del caso: `docs/use-cases/case-h-rag-chatbot.md`.


## 19. Marco teórico (nivel doctoral)

### Retrieval-Augmented Generation (Lewis et al. 2020)

$$
P(y \mid x) = \sum_{z \in \mathcal{Z}} P_\eta(z \mid x) \cdot P_\theta(y \mid x, z)
$$

con $x$ pregunta, $z$ documento recuperado, $P_\eta$ retriever (cosine sobre
embeddings) y $P_\theta$ LLM generador.

### Similarity coseno

$$
\text{sim}(x, z) = \frac{\mathbf{e}_x \cdot \mathbf{e}_z}{\|\mathbf{e}_x\| \|\mathbf{e}_z\|}
$$

### Tools tipadas

$$
\mathcal{T} = \{ t_i : \mathbb{X}_i \to \mathbb{Y}_i \mid \text{schema JSON} \}
$$

Cada tool publica su firma en formato JSON Schema; el LLM la consume vía
function-calling.

### Métricas

$$
\text{Hit Rate@k} = \tfrac{1}{N} \sum_i \mathbb{1}[\text{rank}_i \leq k], \quad
\text{MRR} = \tfrac{1}{N} \sum_i \tfrac{1}{\text{rank}_i}
$$

Objetivos: $\text{Hit@5} \geq 0.85$, $\text{MRR} \geq 0.7$, Faithfulness ≥ 0.9.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

El chatbot es la **cara visible** de CAPTIA al usuario final (profesores, equipo de mantenimiento). Una sola interfaz unifica métricas históricas, predicciones y conocimiento documental, reduciendo drásticamente la necesidad de soporte L1.

### ROI estimado

| Concepto | Valor |
|---|---|
| Reducción tickets soporte L1 | +3 500 €/año |
| Tiempo respuesta profesores | +1 200 €/año |
| **Bruto** | **+4 700 €/año** |
| Coste API LLM (Claude/GPT) | -1 800 €/año |
| **Neto** | **+2 900 €/año** |

### Riesgos y mitigaciones

- Hallucinations del LLM: mitigar con tools de hechos verificables.
- Coste API escala linealmente con uso: monitorizar.


## 21. Bibliografía y referencias

- Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS.
- Reimers, N. & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. EMNLP.
- LangChain Project. *Documentation*. https://python.langchain.com
- Anthropic (2024). *Claude 3.5 Sonnet Model Card*.
